In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
from utils.model_loader import get_model_fits
from utils.sparsity import forward_pass_tanh, local_prune_weights
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from utils.generate_data import generate_correlated_Friedman_data

data_dir      = "datasets/friedman_correlated"
results_dir   = "results/regression/single_layer/tanh/friedman_correlated"
model_names   = ["Gaussian", "RHS", "DHS", "DST"]
sparsity_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

fits = {}
for fname in sorted(f for f in os.listdir(data_dir) if f.endswith(".npz")):
    config = fname.replace(".npz", "")
    fit = get_model_fits(config=config, results_dir=results_dir,
                         models=model_names, include_prior=False)
    if fit:
        fits[config] = fit
print(f"Loaded {len(fits)} configs")


In [ ]:
def build_posterior_mask(W_samples, sparsity):
    """Binary mask: zero out the lowest-|E[w]| fraction of weights."""
    if sparsity == 0.0:
        return np.ones(W_samples.shape[1:], dtype=float)
    score = np.abs(W_samples).mean(axis=0)
    k = int(np.floor(sparsity * score.size))
    thresh = np.partition(score.ravel(), k - 1)[k - 1]
    mask = (score > thresh).astype(float)
    # fix ties to hit exact sparsity
    diff = int(mask.sum()) - (score.size - k)
    if diff > 0:
        tied = np.where(score.ravel() == thresh)[0]
        mask.ravel()[tied[:diff]] = 0.0
    return mask


def evaluate_sparsity(fits, data_func, model_names, sparsity_levels, N_eval=2000):
    """
    For every (config, model, sparsity) compute posterior-mean RMSE on a
    fresh N_eval-point test set drawn with the same seed as training.
    Returns a long-form DataFrame.
    """
    import re
    _KEY = re.compile(r"Friedman_N(\d+)_p\d+_sigma([\d.]+)_seed(\d+)")

    rows = []
    for config, model_fits in fits.items():
        m = _KEY.match(config)
        N, sigma, seed = int(m.group(1)), float(m.group(2)), int(m.group(3))

        # Standardise eval set the same way training did
        _, _, y_tr, _ = data_func(N=N, D=10, sigma=sigma, seed=seed)
        y_mean, y_std = y_tr.mean(), y_tr.std()
        _, X_eval, _, y_eval_raw = data_func(N=N_eval, D=10, sigma=sigma, seed=seed + 999)
        y_eval = (y_eval_raw - y_mean) / y_std

        for model in model_names:
            if model not in model_fits or "posterior" not in model_fits[model]:
                continue
            post = model_fits[model]["posterior"]
            W1 = post.stan_variable("W_1")         # (S, P, H)
            b1 = post.stan_variable("hidden_bias")  # (S, 1, H)
            W2 = post.stan_variable("W_L")          # (S, H, 1)
            b2 = post.stan_variable("output_bias")  # (S,)
            S  = W1.shape[0]

            for q in sparsity_levels:
                mask = build_posterior_mask(W1, q)
                y_hats = np.zeros((S, len(y_eval)))
                for s in range(S):
                    y_hats[s] = forward_pass_tanh(
                        X_eval, W1[s] * mask, b1[s].reshape(-1), W2[s], b2[s].reshape(-1)
                    ).squeeze()
                rmse = float(np.sqrt(np.mean((y_hats.mean(axis=0) - y_eval) ** 2))) * y_std
                rows.append(dict(config=config, N=N, sigma=sigma, seed=seed,
                                 model=model, sparsity=q, rmse=rmse))
    return pd.DataFrame(rows)


def aggregate(df):
    """Mean ± 1 std over seeds, per (N, model, sparsity)."""
    g = df.groupby(["N", "model", "sparsity"])["rmse"]
    return g.agg(center="mean", spread="std").reset_index()


In [ ]:
df_raw = evaluate_sparsity(fits, generate_correlated_Friedman_data, model_names, sparsity_levels)
df_agg = aggregate(df_raw)


In [ ]:
colors    = {"Gaussian": "C0", "RHS": "C1", "DHS": "C2", "DST": "C3"}
model_abbr = {"Gaussian": "Gauss", "RHS": "RHS", "DHS": "DHS", "DST": "DST"}
Ns = [100, 200, 500]
sparsity_plot = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=False)
for c, N in enumerate(Ns):
    ax = axes[c]
    dN = df_agg[df_agg["N"] == N]
    for model in model_names:
        g = dN[dN["model"] == model].sort_values("sparsity")
        g = g[g["sparsity"].isin(sparsity_plot)]
        ax.plot(g["sparsity"], g["center"], lw=2.5, marker="o",
                color=colors[model], label=model_abbr[model])
    ax.set_title(f"N={N}", fontsize=18)
    ax.set_xlabel("Sparsity", fontsize=14)
    if c == 0:
        ax.set_ylabel("RMSE (original scale)", fontsize=14)
    ax.set_xticks(sparsity_plot[::2])
    ax.grid(True, linestyle="--", linewidth=0.5)
    ax.tick_params(labelsize=13)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", frameon=False, fontsize=13)
plt.tight_layout()
plt.savefig("figures_for_use_in_paper/fig17_friedman_sparsity_correlated.pdf", bbox_inches="tight")
plt.show()
